# 70 — Train BGE-M3 bi-encoder (Stage A of nDCG-stretch plan)

Fine-tunes BAAI/bge-m3 on per-music-turn conversation pairs walked
from `talkpl-ai/TalkPlayData-Challenge-Dataset` (train split) via the
Task 6 builder. Uses a custom PEFT-LoRA training loop
(sentence-transformers + peft) — NOT FlagEmbedding's CLI, which
lacks the LoRA flags this plan needs.

**Prereqs**: HF_TOKEN in Colab Secrets. Drive folder
`recsys2026_retrieval_v2_cache` exists.

**Wallclock**: ~8-12 hr on Blackwell (1.5 hr HN mining + 6-10 hr train).

In [ ]:
# 1) Setup — clone + HF auth + Drive mount + deps.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q --upgrade \
    'peft>=0.10' 'transformers>=4.40' 'accelerate>=0.30' \
    'sentence-transformers>=3.0' 'FlagEmbedding>=1.3' \
    'datasets' 'pandas<3.0' 'tqdm' 'omegaconf' 'pyyaml' 'tensorboard'

In [ ]:
# 2) Smoke: build 200 triples to verify the HN miner works end-to-end.
# IMPORTANT: --train-conv-hf walks the HF conversation dataset directly
# (Task 6's `_iter_conversation_turns`). Do NOT pass --train-parquet:
# the W2 parquet schema is (source, session_id, track_id, query,
# code_1..3) and lacks chat_history / current_user_query /
# user_profile_raw / conversation_goal that the builder needs.
!cd /content/recsys2026 && python scripts/build_bi_encoder_training_data.py \
    --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
    --output experiments/cache/retrieval_v2/triples_smoke.jsonl \
    --max-rows 200 --percpos-threshold 0.80 --k-negs 15 \
    2>&1 | tail -20
!wc -l experiments/cache/retrieval_v2/triples_smoke.jsonl
!head -3 experiments/cache/retrieval_v2/triples_smoke.jsonl

In [ ]:
# 3) Full HN mining — ~1.5 hr on Blackwell.
import os
RESULTS_DIR = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
!cd /content/recsys2026 && python -u scripts/build_bi_encoder_training_data.py \
    --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
    --output experiments/cache/retrieval_v2/triples_bge_m3.jsonl \
    --percpos-threshold 0.80 --k-negs 15 --batch-size 64 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/hn_mining_log.txt
!wc -l experiments/cache/retrieval_v2/triples_bge_m3.jsonl

In [ ]:
# 4) Smoke fine-tune: 1 epoch on 500 triples — verify code path works.
!head -500 experiments/cache/retrieval_v2/triples_bge_m3.jsonl > experiments/cache/retrieval_v2/triples_smoke_500.jsonl
!cd /content/recsys2026 && python scripts/train_bi_encoder.py \
    --triples experiments/cache/retrieval_v2/triples_smoke_500.jsonl \
    --output-dir /content/bge_m3_smoke \
    --hub-repo OrRim123/recsys2026-bge-m3-smoke \
    --epochs 1 --logging-steps 10 \
    2>&1 | tail -20
!rm -rf /content/bge_m3_smoke

In [ ]:
# 5) FULL fine-tune: ~6-10 hr on Blackwell. Pushes merged model to Hub.
!cd /content/recsys2026 && python -u scripts/train_bi_encoder.py \
    --triples experiments/cache/retrieval_v2/triples_bge_m3.jsonl \
    --output-dir /content/bge_m3_finetune \
    --hub-repo OrRim123/recsys2026-bge-m3-music-v1 \
    --results-dir /content/drive/MyDrive/recsys2026_retrieval_v2_cache/results/bge_m3 \
    --merge --cleanup-after-push \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/bge_m3_train_log.txt

In [ ]:
# 6) Re-embed the catalog with the fine-tuned model, into DENSE_LOCAL's expected path.
# DENSE_LOCAL reads from {cache_dir}/dense_local/{safe_model}/{embed_label}/track_embeddings.pkl
# We pin embed_label='bge-m3-music-v1-merged' so wRRF factory entry can find it.
import os, pickle, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from mcrs.retrieval_modules.bge_m3_format import format_track_text

HUB_REPO = 'OrRim123/recsys2026-bge-m3-music-v1-merged'
EMBED_LABEL = 'bge-m3-music-v1-merged'
CACHE_ROOT = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local'
safe_model = HUB_REPO.replace('/', '_')
out_dir = os.path.join(CACHE_ROOT, safe_model, EMBED_LABEL)
os.makedirs(out_dir, exist_ok=True)

model = SentenceTransformer(HUB_REPO, device='cuda')
tm = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
texts = [format_track_text(r.get('track_name','unknown'), r.get('artist_name'), r.get('album_name'), r.get('release_date'), r.get('tag_list')) for r in tm]
track_ids = [r['track_id'] for r in tm]
embs = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
embs = np.asarray(embs, dtype=np.float32)
out_path = os.path.join(out_dir, 'track_embeddings.pkl')
with open(out_path, 'wb') as f:
    pickle.dump({'track_ids': track_ids, 'track_mat': embs}, f)
print(f'wrote {len(track_ids)} embeddings → {out_path}')
# Symlink into experiments/cache/dense_local so the runtime cache_dir lookup hits.
local_cache = '/content/recsys2026/experiments/cache/dense_local'
os.makedirs(local_cache, exist_ok=True)
local_link = os.path.join(local_cache, safe_model)
if not os.path.exists(local_link):
    os.symlink(os.path.join(CACHE_ROOT, safe_model), local_link)
print('symlink ready:', local_link)